In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
file_path = '../data/investigator_nacc72.csv'
data = pd.read_csv(file_path, delimiter=',')
pd.set_option('display.max_columns', None)

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
# Show all columns 

print(data.columns.to_list())

In [ ]:
na_counts = data.isna().sum()
missing_data = na_counts[na_counts > 0].sort_values(ascending=False)
print(missing_data)

In [ ]:
age_mean = data['NACCAGE'].mean()
age_median = data['NACCAGE'].median()
age_variance = data['NACCAGE'].var()


print(age_mean)
print(age_median)
print(age_variance)
print(np.sqrt(age_variance))

In [ ]:
unique_ids = data['NACCID'].unique()
print(unique_ids)
unique_visit_number = data['NACCVNUM'].unique()
print(unique_visit_number)

### Häufigkeit der Kategorien des Ratings für die Diagnose

In [ ]:
fig = plt.figure(figsize=(15, 8))
category_counts = data['CDRGLOB'].value_counts()
ax = sns.countplot(
    data=data,
    y=data['CDRGLOB'],
    order=category_counts.index
)
ax.bar_label(ax.containers[0], padding=0.5)
ax.set_title("Barplot of intensity of impairment")
ax.set_xlabel("Amount")
ax.set_ylabel("Intensity")
plt.show()

In [ ]:
fig = plt.figure(figsize=(15, 8))
category_counts = data['NACCUDSD'].value_counts()
ax = sns.countplot(
    data=data,
    y=data['NACCUDSD'],
    order=category_counts.index
)
ax.set_title("Barplot of cognitiv impairment")
ax.set_xlabel("Amount")
ax.set_ylabel("Category")
ax.bar_label(ax.containers[0], padding=0.5)
plt.show()

#### Building of N-Table

In [ ]:
table_columns = [
    'NACCID',
    'NACCVNUM', # How many times visited    
    'NACCAGE',     
    'NACCDAYS', # Days since first visit
    'CDRGLOB',  # Value of severity of alzheimers   
    'NACCMMSE', # Cognition test result   
    'INDEPEND',    
    'RESIDENC'     
]

n_table = data[table_columns].copy()
n_table = n_table.sort_values(by=['NACCID', 'NACCVNUM'])
n_table = n_table.reset_index(drop=True)
n_table['CDRGLOB_diff'] = n_table.groupby('NACCID')['CDRGLOB'].diff()
display(n_table)

In [ ]:
starke_veränderung_maske = (n_table['CDRGLOB_diff'] >= 1.0) & n_table['CDRGLOB'].notna()
patienten_mit_sprung = n_table[starke_veränderung_maske]

exemplarische_ids = patienten_mit_sprung['NACCID'].unique()

if len(exemplarische_ids) > 0:
    exemplarische_id = exemplarische_ids[0]
    print(f"Gefundene exemplarische Patienten-ID mit starker Veränderung: {exemplarische_id}")
    
    print("\nVerlaufsdaten für diesen Patienten:")
    print(n_table[n_table['NACCID'] == exemplarische_id])
else:
    print("Kein Patient mit einem CDRGLOB-Sprung von >= 1.0 in den Daten gefunden.")
    exemplarische_id = None
patient_id = exemplarische_id
data_spec_pers = n_table[n_table['NACCID'] == patient_id]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

sns.lineplot(
    data=data_spec_pers, 
    x='NACCVNUM', 
    y='NACCMMSE', 
    marker='o', 
    ax=ax1, 
    color='blue'
)
ax1.set_title(f'Klinischer Verlauf für Patient {patient_id}')
ax1.set_ylabel('MMSE Score (0-30)')
ax1.set_ylim(0, 32) 
ax1.grid(True, linestyle='--', alpha=0.7)

sns.lineplot(
    data=data_spec_pers, 
    x='NACCVNUM', 
    y='CDRGLOB', 
    marker='o', 
    ax=ax2, 
    color='red'
)
ax2.set_ylabel('CDR Global (0-3)')
ax2.set_xlabel('Visit Nummer (NACCVNUM)')
ax2.set_ylim(-0.2, 3.2)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()